# Part A · Caltech-256 training: ResNet-10t and ResNet-200d (Kaggle)

This notebook only runs `train/hp_tune.py` (LP → FT) for our two sidekicks. Logits, temperature scaling and evaluation run from the `partA-kaggle` notebook once everyone's checkpoints are in.

**Notebook settings:** Accelerator *GPU T4 ×2*, Internet *on* (needs a phone-verified account). Under *Add-ons → Secrets*, add and attach `GH_TOKEN` (GitHub, read access to the repo) and `HF_TOKEN` (Hugging Face, **write** access).

**Where the results go:** the folder `partA-10t-200d/` (`HF_DIR`) in our Hugging Face dataset repo [`emrailklmin/DD2610`](https://huggingface.co/datasets/emrailklmin/DD2610). Nothing outside that folder is changed. While training runs, a background thread uploads `/kaggle/working/partA-10t-200d` (`checkpoints/`, `configs/`, `ray_results/`) every `SYNC_MINUTES`, and again after each model. Download the checkpoints from the repo's *Files* tab on huggingface.co at any time.

**Choosing models:** comment out a line in `MODELS` in the first cell.

**Continuing after an interruption:** just run the notebook again, in any session. It first downloads that folder into `/kaggle/working`, then skips models whose FT checkpoint exists. A model whose LP finished skips LP, and Ray resumes an interrupted search: finished trials are kept and unfinished trials restart from epoch 0. Run it with *Save Version → Save & Run All* so it keeps going with the browser closed (at most 12 h per version).

In [ ]:
from pathlib import Path
import torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
N_GPUS = torch.cuda.device_count()  # 2 on T4 x2
SYNC_MINUTES = 15                   # how often training results are uploaded to Hugging Face

REPO_URL = "https://github.com/gorrast/DD2610-Final-Project.git"
GITHUB_TOKEN_SECRET = "GH_TOKEN"
HF_TOKEN_SECRET = "HF_TOKEN"
HF_REPO = "emrailklmin/DD2610"     # Hugging Face dataset repo for results
HF_DIR = "partA-10t-200d"          # our subfolder in it
OUT = Path('/kaggle/working') / HF_DIR

# Comment out a model to skip it. (model_name, source, tv_weights)
MODELS = [
    ('resnet10t.c3_in1k', 'timm', None),
    ('resnet200d.ra2_in1k', 'timm', None),
]

## 0 · Code and environment

In [ ]:
import os, sys
from base64 import b64encode
from huggingface_hub import HfApi, snapshot_download

auth = b64encode(f"x-access-token:{secrets.get_secret(GITHUB_TOKEN_SECRET)}".encode()).decode()
!rm -rf /tmp/repo && git -c http.extraHeader="Authorization: Basic {auth}" clone -q --depth 1 -b partA-kaggle-10t-200d {REPO_URL} /tmp/repo
%cd /tmp/repo/original_code

# Restore earlier results from the Hugging Face repo, so an interrupted run continues where it stopped.
# Paths in the repo match paths under /kaggle/working, so only HF_DIR is synced and nothing else in the repo is touched.
hf = HfApi(token=secrets.get_secret(HF_TOKEN_SECRET))
SYNCED = [f'{HF_DIR}/{d}/**' for d in ['checkpoints', 'configs', 'ray_results']]
snapshot_download(HF_REPO, repo_type='dataset', local_dir=OUT.parent, allow_patterns=SYNCED, token=hf.token)
print(f'https://huggingface.co/datasets/{HF_REPO}/tree/main/{HF_DIR}')

def sync():
    """Upload OUT to HF_DIR in the repo, mirroring Ray's rotated checkpoints."""
    try:
        hf.upload_folder(repo_id=HF_REPO, repo_type='dataset', folder_path=OUT.parent, allow_patterns=SYNCED,
                         delete_patterns=[f'{HF_DIR}/ray_results/**'], commit_message=f'Sync {HF_DIR} training results')
    except Exception as e:   # e.g. a file Ray deleted mid-upload; the next sync catches up
        print(f'HF sync failed, will retry: {e}')

# The authors' code writes to these relative paths; point them at OUT.
for d in ['checkpoints', 'configs', 'ray_results']:
    (OUT / d).mkdir(parents=True, exist_ok=True)
    os.symlink(OUT / d, d)

# Scripts in train/ import the package folders, so they need original_code on the path.
sys.path.insert(0, os.getcwd())
os.environ.update(PYTHONPATH=os.getcwd(), WANDB_MODE='disabled', PYTHONUNBUFFERED='1')
!pip install -q wilds timm "ray[tune]" fvcore

Optional: check that the token can write to the repo. Uploads a small file, reads it back and deletes it.

In [ ]:
test = f'{HF_DIR}/upload_test.txt'
hf.upload_file(path_or_fileobj=b'upload test', path_in_repo=test, repo_id=HF_REPO, repo_type='dataset')
assert open(hf.hf_hub_download(HF_REPO, test, repo_type='dataset'), 'rb').read() == b'upload test'
hf.delete_file(test, HF_REPO, repo_type='dataset')
print('Upload to', HF_REPO, 'works')

## 1 · Data
Only Caltech-256 is needed for training. It goes to `/tmp/data` on local disk, outside the saved output. The split check uses the authors' seed-42 loader.

In [ ]:
from torchvision.datasets import Caltech256
from load_data import get_dataloaders
from load_models.transforms import get_transforms

DATA = Path('/tmp/data')
Caltech256(DATA, download=True)
caltech = get_dataloaders('caltech256', DATA, 64, 2, get_transforms())
print({k: len(v.dataset) for k, v in caltech.items() if v})   # train 21424, val 4591, test 4592

## 2 · Training config
`caltech256.yaml` is missing from the release. Batch size 128, 8 LP epochs and 16 FT epochs come from paper Appendix A.2. `warmup_epochs` and `grace_period` are **our choice**, because the paper doesn't give them. `num_workers` is split between the GPUs, because two trials share Kaggle's 4 CPUs.

In [ ]:
import yaml

yaml.safe_dump({'training': {'batch_size': 128, 'num_epochs_lp': 8, 'num_epochs_ff': 16,
                             'num_workers': os.cpu_count() // N_GPUS, 'warmup_epochs': 2, 'grace_period': 4}},
               open('configs/caltech256.yaml', 'w'))

**GPU memory check** (about a minute): one FT step for each model in `MODELS`, the way `hp_tune.py` trains: every layer trainable, fp32, timm's default input size, the batch size from the config. LP comes first and takes hours, so an FT out-of-memory error would otherwise show up late. Each check runs in its own process, so the GPU is free again before training.

In [ ]:
import subprocess
CHECK = """
import sys, timm, timm.data, torch
m = timm.create_model(sys.argv[1], num_classes=257).cuda()
x = torch.randn(int(sys.argv[2]), *timm.data.resolve_data_config({}, model=m)['input_size'], device='cuda')
m(x).sum().backward()
torch.optim.AdamW(m.parameters()).step()
print(f'{sys.argv[1]}: peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GB of {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB')
"""
batch = yaml.safe_load(open('configs/caltech256.yaml'))['training']['batch_size']
for name, _, _ in MODELS:
    r = subprocess.run([sys.executable, '-c', CHECK, name, str(batch)], capture_output=True, text=True)
    print(r.stdout.strip() or f'{name}: ' + r.stderr.strip().splitlines()[-1])   # an OutOfMemoryError means FT won't fit

## 3 · LP → FT  →  `train/hp_tune.py`
**Our reduced protocol** (fixed before looking at any test results): a grid search over the learning rate only, with 4 points per stage. Each point runs once. We chose the points from the validation results of the ResNet-18 sweep (`local_3080.ipynb`), around its best LP and FT learning rates and below FT's drop-off between 1.2e-4 and 1.85e-4. All 4 points stay inside the authors' search ranges.
* **LP:** learning rate ∈ {1.4e-3, 1.8e-3, 2.5e-3, 3.4e-3}, weight decay 0.
* **FT:** learning rate ∈ {3.7e-5, 5.9e-5, 9.4e-5, 1.5e-4}, weight decay **fixed at 1e-6** (authors searched [1e-8, 1e-5]).

`hp_tune.py` is unchanged from the authors' release. It builds its search space with `tune.loguniform` and starts it with `tune.run`, both looked up on Ray's `tune` module when called, so the cell below swaps those two functions for our grid before running the script. We start Ray with 4 logical CPUs per GPU, so on T4 ×2 two trials run at once. ASHA can still stop weak FT trials early.

The same swap of `tune.run` also passes `resume="AUTO"`, so Ray continues a search saved in `ray_results/` instead of starting over. Models whose FT checkpoint already exists are skipped.

**Progress and compute time:** a progress bar under the cell counts epochs over all 4 trials, with the latest learning rate, validation accuracy and an ETA. `hp_tune.py` reports only at epochs 0, 3, 6, … so the bar moves in those steps. After each search, `checkpoints/compute_time/<search>.csv` records every trial's GPU hours (one GPU per trial) and is uploaded with the checkpoints.

**Time budget:** our estimate is ~1.2 h for ResNet-10t and ~12 h for ResNet-200d, right at Kaggle's 12 h limit. Watch ResNet-200d's first FT epoch: if it takes more than about 20 min, it won't finish in one version. That's fine now: run the notebook again and it continues from the last upload.

In [ ]:
import ray, runpy, sys, threading, time
import pandas as pd
from tqdm import tqdm
from ray import tune
from ray.tune.tune import run as tune_run   # Ray's own run, so rerunning this cell stays safe

LP_LRS = [1.4e-3, 1.8e-3, 2.5e-3, 3.4e-3]   # chosen from the ResNet-18 LP sweep
FT_LRS = [3.7e-5, 5.9e-5, 9.4e-5, 1.5e-4]   # chosen from the ResNet-18 FT sweep
FT_WD = 1e-6                            # fixed; the authors searched [1e-8, 1e-5]
SPACE = {(1e-4, 1e-2): tune.grid_search(LP_LRS),   # keyed by the ranges written in hp_tune.py
         (1e-6, 3e-4): tune.grid_search(FT_LRS),
         (1e-8, 1e-5): FT_WD}
tune.loguniform = lambda low, high: SPACE[(low, high)]

class Progress(tune.Callback):
    """Bar of epochs done over all trials. hp_tune.py reports at epochs 0, 3, 6, ... so it moves in those steps."""
    def __init__(self, name, n_trials, n_epochs):
        self.bar, self.n_epochs, self.done = tqdm(total=n_trials * n_epochs, desc=name, unit='epoch'), n_epochs, {}
    def advance(self, trial, epochs):
        self.done[trial.trial_id] = epochs
        self.bar.update(sum(self.done.values()) - self.bar.n)
    def on_trial_result(self, trial, result, **info):
        self.bar.set_postfix(lr=trial.config['learning_rate'], val=round(result['val_metric'], 4))
        self.advance(trial, result['training_iteration'] + 1)
    def on_trial_complete(self, trial, **info):   # also when ASHA stops a trial early
        self.advance(trial, self.n_epochs)
    def on_step_begin(self, **info):              # keep the elapsed time ticking between reports
        if time.time() - self.bar.last_print_t > 60: self.bar.refresh()

def run(*args, **kwargs):
    """Ray's tune.run with our grid, resume, a progress bar and a compute-time log."""
    config, name, t0 = kwargs['config'], kwargs['name'], time.time()
    bar = Progress(name, len(config['learning_rate']['grid_search']), config['num_epochs'])
    result = tune_run(*args, **{**kwargs, 'num_samples': 1, 'resume': 'AUTO', 'verbose': 0, 'callbacks': [bar]})
    bar.bar.close()
    times = pd.DataFrame([{'learning_rate': t.config['learning_rate'], 'weight_decay': t.config['weight_decay'],
                           'val_metric': t.last_result.get('val_metric'), 'epochs': t.last_result.get('training_iteration', -1) + 1,
                           'gpu_hours': t.last_result.get('time_total_s', 0) / 3600} for t in result.trials])
    (OUT / 'checkpoints/compute_time').mkdir(exist_ok=True)
    times.to_csv(OUT / 'checkpoints/compute_time' / f'{name}.csv', index=False)
    print(f'{name}: {(time.time() - t0) / 3600:.2f} h wall clock this session, {times.gpu_hours.sum():.2f} GPU h in all trials')
    return result
tune.run = run   # each grid point once; continue a saved search

def sync_every(minutes):
    while True:
        time.sleep(minutes * 60)
        sync()
if not any(t.name == 'hf-sync' for t in threading.enumerate()):
    threading.Thread(target=sync_every, args=(SYNC_MINUTES,), name='hf-sync', daemon=True).start()

!ray stop > /dev/null 2>&1
ray.init(address='local', num_cpus=4 * N_GPUS, num_gpus=N_GPUS, include_dashboard=False)
for name, source, weights in MODELS:
    if (OUT / 'checkpoints/caltech256/ff' / f'{name}.pth').exists():
        print(f'{name}: FT checkpoint exists, skipping')
        continue
    t0 = time.time()
    sys.argv = ['hp_tune.py', '--model_name', name, '--source', source, '--tv_weights', str(weights),
                '--dataset_name', 'caltech256', '--dataset_dir', str(DATA)]
    runpy.run_path('train/hp_tune.py', run_name='__main__')   # runs in this kernel, so the swap applies
    print(f'{name}: {(time.time() - t0) / 3600:.1f} h')
    sync()                                                     # upload this model's results right away